# Predicting Laptop Prices
**Goal**  Estimate the `buynow_price` of a notebook given its specifications.
**Audience**  E-commerce pricing, procurement, competitive-intelligence teams.
**Success metric**  MAE ≤ X USD (define an acceptable error band).


## Table of Contents
* [First Look](#first_look)
    * [Section 2.1](#section_2_1)
        * [Sub Section 2.1.1](#sub_section_2_1_1)
        * [Sub Section 2.1.2](#sub_section_2_1_2)
* [Chapter 3](#chapter3)
    * [Section 3.1](#section_3_1)
        * [Sub Section 3.1.1](#sub_section_3_1_1)
        * [Sub Section 3.1.2](#sub_section_3_1_2)
    * [Section 3.2](#section_3_2)
        * [Sub Section 3.2.1](#sub_section_3_2_1)

### Imports

In [1]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import numpy as np
import ast, joblib, re


from collections import Counter
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import MultiLabelBinarizer
from plotly.subplots import make_subplots
from scipy.stats import spearmanr


from utils.custom_libs import *

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MultiLabelBinarizer, StandardScaler
from sklearn.ensemble import HistGradientBoostingRegressor

pd.options.display.max_rows = None  # Exibir todas as linhas
pd.options.display.max_columns = None  # Exibir todas as colunas
pd.options.display.max_colwidth = None  # Exibir colunas com textos grandes sem truncar


### Creating dataframes

In [2]:
df = pd.read_json("datasets/train_dataset.json")
df_test = pd.read_json("datasets/test_dataset.json")
df_val = pd.read_json("datasets/val_dataset.json")

## First look inside the data <a id="first_look"></a>

##### **Dataset Columns Description**
1. **`graphic card type`**  
   The type of graphics card in the laptop (e.g., integrated, NVIDIA, AMD). Influences the price and performance in tasks like gaming or graphic editing.

2. **`communications`**  
   Communication technologies available in the laptop (e.g., Wi-Fi, Bluetooth, Ethernet).

3. **`resolution (px)`**  
   Screen resolution of the laptop, specified in pixels (e.g., 1920x1080). Relates to display quality.

4. **`CPU cores`**  
   The number of cores in the processor. Affects performance in multitasking and intensive workloads.

5. **`RAM size`**  
   The amount of RAM in the laptop (e.g., 8GB, 16GB), a crucial factor for overall performance.

6. **`operating system`**  
   The operating system installed on the laptop (e.g., Windows 10, macOS, Linux). May affect the price depending on the licensing.

7. **`drive type`**  
   The type of storage in the laptop (e.g., HDD, SSD). SSDs offer faster performance and are usually more expensive.

8. **`input devices`**  
   Input devices included in the laptop (e.g., backlit keyboard, touchpad, fingerprint reader).

9. **`multimedia`**  
   Multimedia features available in the laptop (e.g., speakers, microphones, webcam).

10. **`RAM type`**  
    The type of RAM installed (e.g., DDR3, DDR4), impacting performance and compatibility.

11. **`CPU clock speed (GHz)`**  
    The clock speed of the processor, measured in GHz. Higher speeds generally indicate better performance for single-threaded tasks.

12. **`CPU model`**  
    The specific model of the processor (e.g., Intel Core i5-1135G7, AMD Ryzen 7 5800H). Provides detailed processor capabilities.

13. **`state`**  
    The condition of the laptop (e.g., new, refurbished, used). Directly impacts the price.

14. **`drive memory size (GB)`**  
    The storage capacity in GB (e.g., 256GB, 1TB). Larger capacities often come at a higher price.

15. **`warranty`**  
    The warranty period provided by the manufacturer (e.g., 1 year, 2 years).

16. **`screen size`**  
    The size of the screen, typically measured in inches (e.g., 14", 15.6"). Larger or higher-quality screens may increase the cost.

17. **`buynow_price`**  
    The buy-now price of the laptop. This is the target variable for the prediction model.

In [3]:
print("Rows, columns:", df.shape)
print("Memory usage : {:.1f} MB".format(df.memory_usage(deep=True).sum()/1e6))

df.head()

Rows, columns: (4711, 17)
Memory usage : 4.9 MB


,graphic card type,communications,resolution (px),CPU cores,RAM size,operating system,drive type,input devices,multimedia,RAM type,CPU clock speed (GHz),CPU model,state,drive memory size (GB),warranty,screen size,buynow_price
7233,dedicated graphics,"[bluetooth, lan 10/100/1000 mbps]",1920 x 1080,4,32 gb,[no system],ssd + hdd,"[keyboard, touchpad, illuminated keyboard, numeric keyboard]","[SD card reader, camera, speakers, microphone]",ddr4,2.6,intel core i7,new,1250.0,producer warranty,"17"" - 17.9""",4999.0
5845,dedicated graphics,"[wi-fi, bluetooth, lan 10/100 mbps]",1366 x 768,4,8 gb,[windows 10 home],ssd,"[keyboard, touchpad, numeric keyboard]","[SD card reader, camera, speakers, microphone]",ddr3,2.4,intel core i7,new,256.0,seller warranty,"15"" - 15.9""",2649.0
10303,None,"[bluetooth, nfc (near field communication)]",1920 x 1080,2,8 gb,[windows 10 home],hdd,None,[SD card reader],ddr4,1.6,intel core i7,new,1000.0,producer warranty,"15"" - 15.9""",3399.0
10423,None,None,None,2,None,None,None,None,None,None,NaN,None,new,NaN,producer warranty,None,1599.0
5897,integrated graphics,"[wi-fi, bluetooth]",2560 x 1440,4,8 gb,[windows 10 home],ssd,"[keyboard, touchpad, illuminated keyboard]","[SD card reader, camera, speakers, microphone]",ddr4,1.2,other CPU,new,256.0,producer warranty,"12"" - 12.9""",4499.0


## 0. Raw-look

As we can see here, certain features, such as CPU cores and RAM size, can be converted to a float data type.

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 4711 entries, 7233 to 6037
Data columns (total 17 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   graphic card type       4417 non-null   object 
 1   communications          4261 non-null   object 
 2   resolution (px)         4361 non-null   object 
 3   CPU cores               4711 non-null   object 
 4   RAM size                4457 non-null   object 
 5   operating system        4335 non-null   object 
 6   drive type              4454 non-null   object 
 7   input devices           4321 non-null   object 
 8   multimedia              4310 non-null   object 
 9   RAM type                4212 non-null   object 
 10  CPU clock speed (GHz)   4181 non-null   float64
 11  CPU model               4389 non-null   object 
 12  state                   4711 non-null   object 
 13  drive memory size (GB)  4439 non-null   float64
 14  warranty                4711 non-null   ob

In [5]:
df.describe()

,CPU clock speed (GHz),drive memory size (GB),buynow_price
count,4181.000000,4439.000000,4711.000000
mean,2.342057,652.619284,3495.831195
std,0.386298,467.657354,1727.933306
min,0.800000,0.000000,429.000000
25%,2.100000,250.000000,2222.075000
50%,2.500000,500.000000,3184.000000
75%,2.600000,1000.000000,4399.000000
max,3.900000,2960.000000,15472.650000


Now let's find out how many unique features there are in each column.

In [6]:
contar_valores_unicos(df)

graphic card type            2
communications              97
resolution (px)             12
CPU cores                    6
RAM size                     9
operating system            19
drive type                   5
input devices               12
multimedia                  14
RAM type                     3
CPU clock speed (GHz)       40
CPU model                   18
state                        1
drive memory size (GB)      60
warranty                     3
screen size                  6
buynow_price              1408
dtype: int64

Let's see these unique features

In [7]:
show_unique_values(df)

Column: graphic card type
Total Unique Values: 2
Type: <class 'str'>
Unique Values:
['dedicated graphics' 'integrated graphics']
--------------------------------------------------
Column: communications
Total Unique Values: 97
Type: <class 'str'>
Sample of 10 unique values (and 87 more):
["['bluetooth', 'lan 10/100/1000 mbps']"
 "['wi-fi', 'bluetooth', 'lan 10/100 mbps']"
 "['bluetooth', 'nfc (near field communication)']"
 "['wi-fi', 'bluetooth']" "['wi-fi', 'bluetooth', 'lan 10/100/1000 mbps']"
 "['nfc (near field communication)', 'gps']"
 "['wi-fi 802.11 b/g/n/ac', 'bluetooth', 'lan 10/100/1000 mbps']"
 "['bluetooth', 'lan 10/100 mbps']"
 "['bluetooth', 'lan 10/100/1000 mbps', 'lan 10/100 mbps', 'intel wireless display (widi)', 'nfc (near field communication)', 'modem 3g (wwan)']"
 "['wi-fi 802.11 b/g/n/ac', 'bluetooth', 'lan 10/100 mbps']"]
--------------------------------------------------
Column: resolution (px)
Total Unique Values: 12
Type: <class 'str'>
Sample of 10 unique value

Checking the null values

Analysis of the 4711 entries reveals a substantial presence of null values within the CPU clock speed (GHz) and RAM type columns, both of which are critical for the analytic process. Consequently, a data imputation strategy will be employed. For columns exhibiting nullity below 10%, mode or median imputation will be utilized, with subsequent impact assessment. An alternative methodology involves the creation of an 'unknown' categorical variable.

In [8]:
null_summary = pd.DataFrame({
  "Column": df.columns,
  "Null count": df.isnull().sum(),
  "Percentage Null": ((df.isnull().sum() / len(df)) * 100).round(2)
}).sort_values(by="Null count", ascending=False)

fig = px.bar(null_summary, x="Column", y="Null count",
             title="Null Values by Column", text="Percentage Null",template="plotly_dark")
fig.update_layout(xaxis_title="Columns", yaxis_title="Count of Nulls")
fig.show()

- O histograma mostra que existem mais produtos com faixa de preço entre 2500 e 3000. A tendência central está representada nesses valores, então, poderiamos analisar as caracteristicas desses produtos e traçar estratégias de promoção para produtos mais caros.
- Os dados parecem se manter mais fortemente entre 1500 e 5000, depois decaem muito. Para o modelo, seria bom testar uma tranasfaromação box-cox ou uma log-transformation.
- 

In [9]:
fig = px.histogram(df, x="buynow_price", nbins=40, title="Distribution of buy-now price", template="plotly_dark")
fig.update_layout(xaxis_title="Price", yaxis_title="Frequency")
fig.update()

### Matriz de corelação
Let's see about the correlation between the features. Essa é apenas a primeira versão da matriz de correlação, antes de modelar os dados.

A análise prévia mostra que as colunas, resolution (px), GPU Clock speed, RAM type, CPU Cores e drive types são as mais importantes para a formação do preço do laptop

In [10]:
df_corr = df.copy()
for col in df_corr.columns:
    if df_corr[col].dtype == 'object' or str(df_corr[col].dtype) == 'category' or df_corr[col].dtype == 'bool':
        codes, _ = pd.factorize(df_corr[col], sort=True)
        df_corr[col] = codes.astype('float')
        df_corr.loc[df[col].isna(), col] = np.nan

df_corr = df_corr.loc[:, df_corr.notna().sum() >= 2]

corr = df_corr.corr()

heatmap = go.Heatmap(
    z=corr.values,
    x=corr.columns,
    y=corr.index,
    colorscale='RdBu',
    zmin=-1,
    zmax=1,
    colorbar=dict(title='Correlation')
)
fig = go.Figure(data=[heatmap])
fig.update_layout(
    title='Correlation Matrix (numeric + coded categoricals)',
    template='plotly_dark',
    width=900,
    height=900
)
fig.show()

### Observations

#### Data cleaning
- We have some columns that are not useful for our analysis like 'state', cause have only one value
- See what to do with 'communications' column, there's a lot of values in a list
- Deal with missing values, using the best strategy for each column, none of them aparently need to be dropped
- Change the data type of 'RAM size', 'drive memory size', 'CPU clock speed', 'CPU cores' into numeric columns
- Create separated columns for height and width

#### Data exploration
- Compute summary stats to back the visual: mean, median, std, skewness.
- Flag outliers: list rows above the 99th percentile (≈ R$ 8 k) for manual review.
- Decide on a target transformation (log or √).
- Define price segments for business storytelling and potentially stratified splits.
- Re-draw the histogram on a log-scaled x-axis to show structure inside the dense left half.

## Baseline cleaning

To start, let's get rid of the 'state' column since it only has one value, which won't contribute to our analysis.

In [11]:
df.drop(columns=['state'], inplace=True)

### Column "communications"

In [12]:


# ------------------------------------------------------------------
# 1. helper: normalise a single token
# ------------------------------------------------------------------
_token_re = re.compile(r'\s+')                   # collapse whitespace

def normalise_token(tok: str) -> str:
    tok = tok.lower().strip()

    # remove "(...)" including parentheses
    tok = re.sub(r'\(.*?\)', '', tok)

    # drop everything except letters, numbers and spaces
    tok = re.sub(r'[^0-9a-z\s]', ' ', tok)

    # collapse multiple spaces → single underscore
    tok = _token_re.sub('_', tok).strip('_')

    return tok

# ------------------------------------------------------------------
# 2. Parse and clean the `communications` column
# ------------------------------------------------------------------
comm_series_raw = (
    df['communications']          # e.g. "['wi-fi', 'bluetooth']"
      .fillna('[]')
      .map(ast.literal_eval)      # safe string → list
)

comm_series_clean = comm_series_raw.apply(
    lambda lst: [normalise_token(t) for t in lst if normalise_token(t)]
)

# ------------------------------------------------------------------
# 3. Multi-hot encoding
# ------------------------------------------------------------------
mlb = MultiLabelBinarizer(sparse_output=False)
comm_matrix = mlb.fit_transform(comm_series_clean)

comm_df = pd.DataFrame(
    comm_matrix,
    columns=[f'comm_{tok}' for tok in mlb.classes_],
    index=df.index
)

# ------------------------------------------------------------------
# 4. Combine with original data (and drop raw text column if desired)
# -------------------------------------------------------------------
df_cleaning = pd.concat([df.drop(columns=['communications']), comm_df], axis=1)

print("Shape after merge:", df_cleaning.shape)
print("New comm_* columns:", comm_df.columns.tolist())
display(df_cleaning.head())


Shape after merge: (4711, 28)
New comm_* columns: ['comm_bluetooth', 'comm_gps', 'comm_intel_wireless_display', 'comm_lan_10_100_1000_mbps', 'comm_lan_10_100_mbps', 'comm_modem_3g', 'comm_modem_4g', 'comm_nfc', 'comm_wi_fi', 'comm_wi_fi_802_11_a_b_g_n', 'comm_wi_fi_802_11_a_b_g_n_ac', 'comm_wi_fi_802_11_b_g_n', 'comm_wi_fi_802_11_b_g_n_ac']


,graphic card type,resolution (px),CPU cores,RAM size,operating system,drive type,input devices,multimedia,RAM type,CPU clock speed (GHz),CPU model,drive memory size (GB),warranty,screen size,buynow_price,comm_bluetooth,comm_gps,comm_intel_wireless_display,comm_lan_10_100_1000_mbps,comm_lan_10_100_mbps,comm_modem_3g,comm_modem_4g,comm_nfc,comm_wi_fi,comm_wi_fi_802_11_a_b_g_n,comm_wi_fi_802_11_a_b_g_n_ac,comm_wi_fi_802_11_b_g_n,comm_wi_fi_802_11_b_g_n_ac
7233,dedicated graphics,1920 x 1080,4,32 gb,['no system'],ssd + hdd,"['keyboard', 'touchpad', 'illuminated keyboard', 'numeric keyboard']","['SD card reader', 'camera', 'speakers', 'microphone']",ddr4,2.6,intel core i7,1250.0,producer warranty,"17"" - 17.9""",4999.0,1,0,0,1,0,0,0,0,0,0,0,0,0
5845,dedicated graphics,1366 x 768,4,8 gb,['windows 10 home'],ssd,"['keyboard', 'touchpad', 'numeric keyboard']","['SD card reader', 'camera', 'speakers', 'microphone']",ddr3,2.4,intel core i7,256.0,seller warranty,"15"" - 15.9""",2649.0,1,0,0,0,1,0,0,0,1,0,0,0,0
10303,None,1920 x 1080,2,8 gb,['windows 10 home'],hdd,None,['SD card reader'],ddr4,1.6,intel core i7,1000.0,producer warranty,"15"" - 15.9""",3399.0,1,0,0,0,0,0,0,1,0,0,0,0,0
10423,None,None,2,None,None,None,None,None,None,NaN,None,NaN,producer warranty,None,1599.0,0,0,0,0,0,0,0,0,0,0,0,0,0
5897,integrated graphics,2560 x 1440,4,8 gb,['windows 10 home'],ssd,"['keyboard', 'touchpad', 'illuminated keyboard']","['SD card reader', 'camera', 'speakers', 'microphone']",ddr4,1.2,other CPU,256.0,producer warranty,"12"" - 12.9""",4499.0,1,0,0,0,0,0,0,0,1,0,0,0,0


### Column "RAM size"

In [13]:
df_cleaning['ram_gb'] = (
    df_cleaning['RAM size'].str.extract(r'(\d+)')[0]
              .astype('Int64')
)
display(df_cleaning[['RAM size', 'ram_gb']].head())

,RAM size,ram_gb
7233,32 gb,32
5845,8 gb,8
10303,8 gb,8
10423,None,<NA>
5897,8 gb,8


### Column "CPU clock speed"

In [14]:
df_cleaning['cpu_clock_ghz'] = pd.to_numeric(df_cleaning['CPU clock speed (GHz)'], errors='coerce')

In [15]:
display(df_cleaning.cpu_clock_ghz.unique(),
df_cleaning['CPU clock speed (GHz)'].unique())

array([2.6 , 2.4 , 1.6 ,  nan, 1.2 , 2.  , 2.5 , 2.8 , 1.9 , 1.1 , 2.3 ,
       2.24, 1.8 , 3.  , 1.7 , 2.1 , 2.7 , 1.33, 2.2 , 3.9 , 2.16, 2.9 ,
       1.83, 0.9 , 1.  , 1.44, 1.58, 1.35, 3.1 , 0.8 , 3.5 , 1.3 , 1.5 ,
       1.86, 3.3 , 1.15, 1.68, 1.4 , 2.66, 2.13, 1.66])

array([2.6 , 2.4 , 1.6 ,  nan, 1.2 , 2.  , 2.5 , 2.8 , 1.9 , 1.1 , 2.3 ,
       2.24, 1.8 , 3.  , 1.7 , 2.1 , 2.7 , 1.33, 2.2 , 3.9 , 2.16, 2.9 ,
       1.83, 0.9 , 1.  , 1.44, 1.58, 1.35, 3.1 , 0.8 , 3.5 , 1.3 , 1.5 ,
       1.86, 3.3 , 1.15, 1.68, 1.4 , 2.66, 2.13, 1.66])

### Column "resolution (px)"

In [16]:
w_h = df_cleaning['resolution (px)'].str.extract(r'(\d+)\s*[xX]\s*(\d+)').astype('Int64')
w_h.columns = ['width_px', 'height_px']
df_cleaning = pd.concat([df_cleaning, w_h], axis=1)
# df_cleaning['total_pixels']  = df_cleaning['width_px'] * df_cleaning['height_px']
# df_cleaning['aspect_ratio']  = (df_cleaning['width_px'] / df_cleaning['height_px']).round(2)

### Screen size

In [17]:
df_cleaning['screen_in'] = (
    df_cleaning['screen size']
      .str.extract(r'(\d+\.?\d*)"\s*-\s*(\d+\.?\d*)')[ [0,1] ]
      .astype(float)
      .mean(axis=1)
)


### Drive memory size

Nesse caso existem alguns aproches para tratar esse campo, mas para isso precisamos 

In [18]:
def parse_storage(row):
    txt = f"{row.get('drive memory size (GB)', '')} {row.get('drive type', '')}".lower()
    m = re.search(r'(\d+)\s*gb\s*ssd.*?(\d+)\s*(?:gb|tb)\s*hdd', txt)
    if m:
        ssd = int(m.group(1)); hdd = int(m.group(2)) * (1024 if 'tb' in txt else 1)
        return ssd, hdd
    if 'ssd' in txt and 'hdd' not in txt:
        num = re.search(r'(\d+)', txt); return (int(num[0]) if num else np.nan), 0
    if 'hdd' in txt and 'ssd' not in txt:
        num = re.search(r'(\d+)', txt); return 0, (int(num[0]) if num else np.nan)
    if 'ssd' in txt and 'hdd' in txt:
        num = re.search(r'(\d+)', txt); return (int(num[0]) if num else np.nan), np.nan
    return np.nan, np.nan

df[['ssd_gb','hdd_gb']] = df.apply(parse_storage, axis=1, result_type='expand')
df['price']   = pd.to_numeric(df['buynow_price'], errors='coerce')
df['has_ssd'] = (df['ssd_gb'].fillna(0) > 0).astype(int)

# ---------- correlação ----------
mask   = df['ssd_gb'].notna() & df['price'].notna()
rho, p = spearmanr(df.loc[mask,'ssd_gb'], df.loc[mask,'price'])
print(f'Spearman ρ = {rho:.2f}  (p={p:.3g})')

# ---------- plot Scatter Plotly ----------
fig = px.scatter(df.loc[mask], x='ssd_gb', y='price',
                 title='Preço vs Capacidade de SSD',
                 labels={'ssd_gb':'SSD (GB)', 'price':'Preço (R$)'},
                 trendline='ols')        # linha de tendência opcional
fig.update_layout(template='plotly_dark')
fig.show()

Spearman ρ = 0.55  (p=0)


In [19]:
# def split_storage(mem_gb, drive_type):
#     if 'ssd' in drive_type and 'hdd' in drive_type:
#         # assume half/half if sizes aren't specified separately
#         return mem_gb/2, mem_gb/2
#     return (mem_gb, 0) if 'ssd' in drive_type else (0, mem_gb)

# df_cleaning[['ssd_gb', 'hdd_gb']] = [
#     split_storage(row['drive memory size (GB)'], str(row['drive type']).lower())
#     for _, row in df_cleaning[['drive memory size (GB)', 'drive type']].iterrows()
# ]

def parse_storage(row):
    txt = f"{row['drive memory size (GB)']} {row['drive type']}".lower()

    # explicit "512 gb ssd + 1 tb hdd" (two numbers)
    m = re.search(r'(\d+)\s*gb\s*ssd.*?(\d+)\s*(?:gb|tb)\s*hdd', txt)
    if m:
        ssd = int(m.group(1))
        hdd = int(m.group(2)) * (1024 if 'tb' in txt else 1)
        return ssd, hdd

    # single number tagged as ssd only
    if 'ssd' in txt and 'hdd' not in txt:
        m = re.search(r'(\d+)', txt)
        return (int(m.group(1)) if m else np.nan), 0

    # single number tagged as hdd only
    if 'hdd' in txt and 'ssd' not in txt:
        m = re.search(r'(\d+)', txt)
        return 0, (int(m.group(1)) if m else np.nan)

    # hybrid but ambiguous size: treat ssd as present, leave hdd unknown
    if 'ssd' in txt and 'hdd' in txt:
        m = re.search(r'(\d+)', txt)
        return (int(m.group(1)) if m else np.nan), np.nan

    # no drive info
    return np.nan, np.nan

# apply safely
df_cleaning[['ssd_gb', 'hdd_gb']] = df_cleaning.apply(parse_storage, axis=1, result_type='expand')

# derive helper features
df_cleaning['storage_total_gb'] = df_cleaning[['ssd_gb', 'hdd_gb']].sum(axis=1, min_count=1)
df_cleaning['has_ssd']          = (df_cleaning['ssd_gb'].fillna(0) > 0).astype(int)


In [20]:
df_cleaning[['ssd_gb', 'hdd_gb','storage_total_gb', 'has_ssd', 'drive memory size (GB)','drive type']].head(20)
# df_cleaning

,ssd_gb,hdd_gb,storage_total_gb,has_ssd,drive memory size (GB),drive type
7233,1250.0,NaN,1250.0,1,1250.0,ssd + hdd
5845,256.0,0.0,256.0,1,256.0,ssd
10303,0.0,1000.0,1000.0,0,1000.0,hdd
10423,NaN,NaN,NaN,0,NaN,None
5897,256.0,0.0,256.0,1,256.0,ssd
4870,0.0,1000.0,1000.0,0,1000.0,hdd
2498,0.0,1000.0,1000.0,0,1000.0,hdd
6220,256.0,0.0,256.0,1,256.0,ssd
10594,NaN,NaN,NaN,0,500.0,None
11640,256.0,0.0,256.0,1,256.0,ssd
